# Residual dust in quiescent galaxies: the "ISM equilibrium at large radii" hypothesis

**Working hypothesis.** Some quiescent galaxies keep a residual dust reservoir in
their ISM because that ISM stayed in *equilibrium at large radii* and was **never
funnelled inward to feed the AGN**. Without a strong inflow → no AGN-driven gas
heating → no thermal/sputtering dust destruction → dust survives long after
quenching.

If true, we expect (relative to galaxies that quenched the "violent" way):

- the surviving cold-gas / dust reservoir is **spatially extended** (large
  half-mass radius, *not* centrally concentrated) through and after quenching;
- the reservoir is **not consumed/destroyed rapidly**, so M$_{\rm dust}$, M$_{\rm H_2}$
  and M$_{\rm HI}$ decline gently;
- this is preferentially the case for **slow quenchers** (long $\tau_q$), whereas
  **fast quenchers** drive gas to the centre, feed the AGN, and destroy dust.

### Sample & method (agreed design)

- **Sample:** every galaxy *passive at z = 0*, defined by $\mathrm{sSFR}(z{=}0) < 0.2/t_H$
  (the same $0.2/t$ threshold used to mark the quenching time `QT`), **and**
  $\log_{10}(M_\star/M_\odot) > 10.5$. Residual dust is a *measured outcome*, never
  a selection cut.
- **Per-galaxy evolutionary anchors** (not absolute snapshots). For each galaxy we
  trace its progenitors and locate, on its own track:
  `SF-peak` (max sSFR) · `SFT` (slow-formation crossing of $1/t$) ·
  `QT` (quenching crossing of $0.2/t$) · `post-quench` (persistence end) ·
  `dust-peak` · `dust-min` (minimum dust before it hits ~0) · `z=0`.
- **"Star-forming" reference:** the *same* galaxies sampled at their **SFT**, i.e.
  when they were still forming stars (no separate control sample).
- **Fast vs slow:** $\tau_q = t_{QT} - t_{SFT}$, split at **2×10⁸ yr**; we also report
  $\tau_q / t_H(z_{QT})$ for context.
- **Part 1 — catalogue only** (CAESAR integrated $M_{\rm dust}, M_{H_2}, M_{HI}$, radii):
  distributions at each anchor, fast vs slow.
- **Part 2 — particle files**: face-on radial $\Sigma$ profiles → effective radii
  $R_{50}, R_{90}$ of dust/H₂/HI → **does the dust extension vary with time for fast
  vs slow quenchers?**

> **Run environment.** This repository's data live on the cluster; the heavy steps
> (catalogue loads, progenitor build, particle extraction) are gated behind
> `BUILD_*` / `RUN_*` flags so the notebook can be re-run incrementally there.

## 0 · Setup & configuration

In [ ]:
import os
import gc
import numpy as np
import h5py
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.cosmology import Planck15 as COSMO   # matches the repo's quenching batch

from simbanator.io.simba import Simulation
from simbanator.analysis import HDF5BuildHistory, caesar_read_progen, extract_particles
from simbanator.analysis.quenching import find_quenching_times
from simbanator.utils.geometry import shrink_center, principal_axes, rotate_to_frame

# ── configuration ──────────────────────────────────────────────────────────
SIM_NAME      = "cis100"          # configured simulation name (~/.simbanator/config.json)
BOX           = 100               # Mpc/h, used for sanity only
START_SNAP    = 150               # z ~ 0 anchor for the tracks (snap 151 == z=0; 150 ~ z<0.01)
END_SNAP      = 44                # z ~ 4.8, far enough back to capture SF-peak/SFT
PROG_RANGE    = range(END_SNAP, START_SNAP + 1)   # snaps used to build the progenitor tree

MASS_FLOOR    = 9.5               # log10(M*/Msun) floor for the z=0 sample
MASS_BIN_EDGES  = [9.5, 10.5, 11.0, np.inf]   # log10(M*/Msun) bin edges (binned at z=0)
MASS_BIN_LABELS = ["9.5-10.5", "10.5-11", ">=11"]
PASSIVE_FACTOR = 0.2             # passive if sSFR < PASSIVE_FACTOR / t_H  (== QT threshold)
TAU_SPLIT_YR  = 2.0e8            # fast (<) vs slow (>) quencher boundary on tau_q = QT - SFT
NGAS_MIN      = 50               # min gas particles for catalogue reliability / particle work
CORRUPT_SNAPS = set()            # snapshot numbers known to be truncated/corrupt; auto-grows at runtime

# heavy-step switches (set True the first time, then reuse cached products)
BUILD_PROGEN  = False            # build the progenitor FITS (needs all caesar catalogs)
BUILD_HISTORY = False            # build the property history HDF5 (needs all caesar catalogs)
RUN_PARTICLES = False            # extract particles + build radial profiles (heaviest step)
BUILD_BH      = False            # build the BH accretion / AGN-energy history (reads all catalogs)

# cached product locations (under ./output/<sim>/...)
PROG_FILE     = "progenitors_passive_z0.fits"
HIST_FILE     = "history_passive_z0.hdf5"     # written under output/<sim>/caesar_sfh/

sim   = Simulation(SIM_NAME)
OUT   = os.path.join(os.getcwd(), "output", SIM_NAME)
SFHDIR = os.path.join(OUT, "caesar_sfh")
PLOTDIR = os.path.join(OUT, "plots", "residual_dust")
os.makedirs(PLOTDIR, exist_ok=True)
HIST_PATH = os.path.join(SFHDIR, HIST_FILE)

print("simulation :", SIM_NAME)
print("track snaps:", START_SNAP, "->", END_SNAP, f"(z {sim.get_z_from_snap(START_SNAP):.3f} -> {sim.get_z_from_snap(END_SNAP):.3f})")
print("z=0 t_H    :", f"{COSMO.age(0).value:.3f} Gyr  -> passive cut sSFR < {PASSIVE_FACTOR/ (COSMO.age(0).value*1e9):.2e} /yr")

### 0.1 · Discover the exact CAESAR dataset names

CAESAR stores galaxy scalars under `galaxy_data/dicts/`. Atomic/molecular keys are
usually `masses.HI` and `masses.H2`, radii `radii.stellar_half_mass` /
`radii.gas_half_mass` — but names vary between catalog builds. Run this once to
confirm the keys used below; adjust `PROPS` in §2 if they differ.

In [ ]:
_probe_snap = START_SNAP
_probe_file = sim.get_caesar_file(_probe_snap)
print("probing:", _probe_file)
with h5py.File(_probe_file, "r") as f:
    dnames = sorted(f["galaxy_data/dicts"].keys())
    print("\n-- galaxy_data/dicts keys --")
    for k in dnames:
        print("  ", k)
    print("\n-- galaxy_data datasets --")
    for k in sorted(f["galaxy_data"].keys()):
        if k != "dicts":
            print("  ", k)

## 1 · Build / load progenitor tracks and the property history

We follow the repository's established pipeline
(`caesar_read_progen` → `HDF5BuildHistory`). The progenitor table is built for **all**
z≈0 galaxies; we then read the full property history and select the passive,
massive subset from the first time-row (§2). `dust`, `H2`, `HI`, gas, stellar mass,
both half-mass radii and positions are all pulled in one pass.

In [ ]:
# ── 1a. progenitor tree (optional, heavy) ──────────────────────────────────
if BUILD_PROGEN:
    cs0 = sim.load_catalog(snap=START_SNAP)
    all_ids = [g.GroupID for g in cs0.galaxies]
    caesar_read_progen(all_ids, PROG_FILE, PROG_RANGE, sim, output_dir=None)
    del cs0; gc.collect()
    print("progenitor table written.")
else:
    print("BUILD_PROGEN=False -> using existing", os.path.join(OUT, "progenitors", PROG_FILE))

In [ ]:
# ── 1b. property history (optional, heavy) ─────────────────────────────────
# NOTE: confirm these keys against the §0.1 probe. 'masses.HI' / 'masses.H2' are
# the standard CAESAR-SIMBA names; change here if your build differs.
PROPS = {
    "galaxy_data": [
        "masses.stellar", "sfr", "masses.gas", "masses.dust",
        "masses.H2", "masses.HI",
        "radii.stellar_half_mass", "radii.gas_half_mass", "radii.dust_half_mass",
        "pos", "ngas", "nstar",
    ],
    "halo_data": ["masses.total"],
}

if BUILD_HISTORY:
    cs0 = sim.load_catalog(snap=START_SNAP)
    hist = HDF5BuildHistory(sim, cs0, progfilename=PROG_FILE)
    with fits.open(hist.progen_file) as hdul:
        valid_ids = np.asarray(hdul[1].data["GroupID"])
    hist.get_history_indx(valid_ids, START_SNAP, END_SNAP)
    # robust read: drop any property the catalog build doesn't have (e.g. a missing
    # radii.dust_half_mass or masses.HI) and retry, instead of failing the whole build.
    props_try = {k: list(v) for k, v in PROPS.items()}
    while True:
        try:
            z_dict, props = hist.get_property_history(props_try, verbose=1)
            break
        except KeyError as e:
            msg = str(e); dropped = False
            for fam, plist in props_try.items():
                for pr in list(plist):
                    if pr in msg or pr.split("/")[-1] in msg:
                        plist.remove(pr); print("  [history] dropping missing property:", pr); dropped = True
            if not dropped:
                raise
    hist.save_history_to_hdf5(HIST_FILE)
    del cs0; gc.collect()
    print("history written to", HIST_PATH)

In [ ]:
# ── 1c. load the cached history ────────────────────────────────────────────
with h5py.File(HIST_PATH, "r") as f:
    galaxy_ids = f["metadata/galaxy_ids"][:]                 # (n_gal,) GroupID at START_SNAP
    snaps_arr  = f["metadata/snapshots"][:]                  # (n_snap,) snapshot numbers
    redshift   = f["redshift/Redshift"][:]                   # (n_snap,)
    P = {k: f[f"properties/{k}"][:] for k in f["properties"].keys()}

# arrays are ordered from START_SNAP (row 0, ~z=0) down to END_SNAP
t_cosmic_yr = COSMO.age(redshift).value * 1e9                # (n_snap,) cosmic time [yr]

n_snap, n_gal = P["sfr"].shape
print(f"history: {n_snap} snapshots x {n_gal} galaxies")
print("row 0  -> snap", snaps_arr[0], f"z={redshift[0]:.4f}")
print("row -1 -> snap", snaps_arr[-1], f"z={redshift[-1]:.4f}")
print("properties:", list(P.keys()))

## 2 · Define the z=0 passive, massive sample

Selection uses the first time-row of the history (so indices stay aligned with the
tracks). Passive ≡ $\mathrm{sSFR} < 0.2/t_H$ at that epoch; massive ≡
$\log_{10}M_\star > 10.5$; plus a gas-particle floor so later particle work is meaningful.

In [ ]:
mstar0 = P["masses.stellar"][0]
sfr0   = P["sfr"][0]
ngas0  = P["ngas"][0]
ssfr0  = np.where(mstar0 > 0, sfr0 / mstar0, np.nan)

tH0       = t_cosmic_yr[0]                       # cosmic time at the z~0 anchor [yr]
passive   = ssfr0 < (PASSIVE_FACTOR / tH0)
massive   = np.log10(np.where(mstar0 > 0, mstar0, np.nan)) > MASS_FLOOR
enoughgas = ngas0 >= NGAS_MIN

sample_mask = passive & massive & enoughgas
sample_cols = np.where(sample_mask)[0]           # column indices into the history arrays
sample_gids = galaxy_ids[sample_cols]            # GroupIDs at START_SNAP

print(f"passive            : {np.nansum(passive)}")
print(f"+ logM* > {MASS_FLOOR}     : {np.nansum(passive & massive)}")
print(f"+ ngas >= {NGAS_MIN}      : {sample_mask.sum()}  <- final sample")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(np.log10(mstar0), np.log10(ssfr0), s=4, c="0.7", label="all z~0")
ax[0].scatter(np.log10(mstar0[sample_cols]), np.log10(ssfr0[sample_cols]), s=8, c="C3", label="sample")
ax[0].axhline(np.log10(PASSIVE_FACTOR/tH0), ls="--", c="k")
for _e in MASS_BIN_EDGES[:-1]:
    ax[0].axvline(_e, ls=":", c="k", lw=1)        # mass-bin boundaries (9.5, 10.5, 11)
ax[0].set_xlabel(r"$\log_{10} M_\star$"); ax[0].set_ylabel(r"$\log_{10}$ sSFR [yr$^{-1}$]"); ax[0].legend()
dts = P["masses.dust"][0] / np.where(mstar0 > 0, mstar0, np.nan)
ax[1].hist(np.log10(dts[sample_cols][dts[sample_cols] > 0]), bins=25, color="C3", alpha=0.8)
ax[1].set_xlabel(r"$\log_{10}(M_{\rm dust}/M_\star)$ at z$\approx$0"); ax[1].set_ylabel("N")
ax[1].set_title("residual dust in the passive sample")
plt.tight_layout(); plt.show()

## 3 · Per-galaxy critical evolutionary points & fast/slow classification

For each sampled galaxy we run `find_quenching_times` on its sSFR(t) track to get
**SFT**, **QT**, and the **persistence end** (post-quench). When several quench
events exist we keep the **latest** one (the one that leaves the galaxy passive at
z=0). We add **SF-peak** (max sSFR), **dust-peak**, and **dust-min** (minimum dust
before the track first drops to ~0). Each critical *time* is mapped to its nearest
snapshot row, and we record the galaxy's CAESAR index there for later particle work.

$\tau_q = t_{QT}-t_{SFT}$ → **fast** if $\tau_q < 2\times10^8$ yr, else **slow**;
$\tau_q/t_H(z_{QT})$ is stored alongside.

In [ ]:
STAGES = ["sf_peak", "sft", "qt", "post_quench", "dust_peak", "dust_min", "z0"]

def _nearest_row(t_target):
    if not np.isfinite(t_target):
        return -1
    return int(np.argmin(np.abs(t_cosmic_yr - t_target)))

def _dust_min_before_zero(dust, valid):
    '''Cosmic time of minimum dust mass, restricted to before dust first hits ~0.'''
    d = np.where(valid, dust, np.nan)
    nz = np.isfinite(d) & (d > 0)
    if nz.sum() < 2:
        return np.nan
    idx = np.where(nz)[0]
    # first time (going forward in cosmic time) the *valid* dust track collapses to ~0
    order_t = idx[np.argsort(t_cosmic_yr[idx])]
    dvals = d[order_t]
    floor = 1e-6 * np.nanmax(dvals)
    cut = np.argmax(dvals <= floor) if np.any(dvals <= floor) else len(dvals)
    cut = max(cut, 2)
    seg = order_t[:cut]
    return t_cosmic_yr[seg[np.argmin(d[seg])]]

records = []
for col, gid in zip(sample_cols, sample_gids):
    mstar = P["masses.stellar"][:, col]
    sfr   = P["sfr"][:, col]
    dust  = P["masses.dust"][:, col]
    ssfr  = np.where(mstar > 0, sfr / mstar, np.nan)
    valid = np.isfinite(ssfr) & (ssfr > 0) & np.isfinite(t_cosmic_yr)
    if valid.sum() < 5:
        continue

    t = t_cosmic_yr[valid]; s = ssfr[valid]
    o = np.argsort(t); t, s = t[o], s[o]
    tu, ui = np.unique(t, return_index=True); su = s[ui]
    if len(tu) < 5:
        continue

    qts, sfts, _, dbg = find_quenching_times(
        tu, su, galaxy_id=int(gid), plot=False, save_fits_path=None, return_debug=True,
    )
    if len(qts) == 0:
        continue
    k = int(np.argmax(qts))                          # latest quench event -> the z=0 one
    t_qt, t_sft = qts[k], sfts[k]
    t_postq = dbg["persistence_end_times"][k]

    # other anchors
    t_sfpeak  = t_cosmic_yr[np.nanargmax(np.where(valid, ssfr, -np.inf))]
    t_dpeak   = t_cosmic_yr[np.nanargmax(np.where(valid, dust, -np.inf))] if np.any(valid & (dust > 0)) else np.nan
    t_dmin    = _dust_min_before_zero(dust, valid)
    t_z0      = t_cosmic_yr[0]

    rows = {st: _nearest_row(tt) for st, tt in zip(
        STAGES, [t_sfpeak, t_sft, t_qt, t_postq, t_dpeak, t_dmin, t_z0])}

    tau_q = t_qt - t_sft
    z_qt  = float(np.interp(t_qt, t_cosmic_yr[::-1], redshift[::-1]))
    tH_qt = COSMO.age(z_qt).value * 1e9
    records.append(dict(
        gid=int(gid), col=int(col),
        t_sf_peak=t_sfpeak, t_sft=t_sft, t_qt=t_qt, t_post_quench=t_postq,
        t_dust_peak=t_dpeak, t_dust_min=t_dmin, t_z0=t_z0,
        tau_q=tau_q, tau_q_over_tH=tau_q/tH_qt, z_qt=z_qt,
        **{f"row_{st}": rows[st] for st in STAGES},
    ))

print(f"galaxies with a usable quench event: {len(records)} / {len(sample_cols)}")

In [ ]:
import numpy.lib.recfunctions as rfn

# assemble a structured table of critical points + class
gids   = np.array([r["gid"] for r in records])
cols   = np.array([r["col"] for r in records])
tau_q  = np.array([r["tau_q"] for r in records])
tau_n  = np.array([r["tau_q_over_tH"] for r in records])
is_fast = tau_q < TAU_SPLIT_YR
classes = np.where(is_fast, "fast", "slow")

# stellar-mass bin (at z=0) for every record -> used to facet ALL comparisons below
NBINS = len(MASS_BIN_LABELS)
logm_rec = np.log10(np.where(P["masses.stellar"][0, cols] > 0, P["masses.stellar"][0, cols], np.nan))
mbin = np.full(len(records), -1, dtype=int)
for _b in range(NBINS):
    inb = (logm_rec >= MASS_BIN_EDGES[_b]) & (logm_rec < MASS_BIN_EDGES[_b + 1])
    mbin[inb] = _b

print(f"fast quenchers (tau_q < {TAU_SPLIT_YR:.1e} yr): {is_fast.sum()}")
print(f"slow quenchers                               : {(~is_fast).sum()}")
print("mass bins (logM* at z=0):")
for _b in range(NBINS):
    bm = mbin == _b
    print(f"  {MASS_BIN_LABELS[_b]:9s}: {bm.sum():4d}  (fast {np.sum(bm & is_fast)}, slow {np.sum(bm & ~is_fast)})")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(np.log10(tau_q[tau_q > 0]), bins=20, color="C0", alpha=0.8)
ax[0].axvline(np.log10(TAU_SPLIT_YR), ls="--", c="k", label=r"$2\times10^8$ yr")
ax[0].set_xlabel(r"$\log_{10}\,\tau_q$ [yr]"); ax[0].set_ylabel("N"); ax[0].legend()
ax[1].hist(np.log10(tau_n[tau_n > 0]), bins=20, color="C1", alpha=0.8)
ax[1].set_xlabel(r"$\log_{10}\,(\tau_q / t_H(z_{QT}))$"); ax[1].set_ylabel("N")
plt.tight_layout(); plt.show()

In [ ]:
# Persist the critical-point catalogue for reuse / TOPCAT
def save_critical_table():
    names = (["GALAXY_ID", "COL", "CLASS", "TAU_Q_YR", "TAU_Q_OVER_TH", "Z_QT"]
             + [f"T_{s.upper()}" for s in STAGES]
             + [f"ROW_{s.upper()}" for s in STAGES])
    coldefs = [
        fits.Column(name="GALAXY_ID", format="K", array=gids),
        fits.Column(name="COL",       format="K", array=cols),
        fits.Column(name="CLASS",     format="8A", array=np.array(classes, dtype="S8")),
        fits.Column(name="TAU_Q_YR",  format="D", array=tau_q),
        fits.Column(name="TAU_Q_OVER_TH", format="D", array=tau_n),
        fits.Column(name="Z_QT",      format="D", array=np.array([r["z_qt"] for r in records])),
    ]
    for s in STAGES:
        coldefs.append(fits.Column(name=f"T_{s.upper()}", format="D",
                                   array=np.array([r[f"t_{s}"] for r in records])))
    for s in STAGES:
        coldefs.append(fits.Column(name=f"ROW_{s.upper()}", format="J",
                                   array=np.array([r[f"row_{s}"] for r in records], dtype=np.int32)))
    outp = os.path.join(OUT, "residual_dust_critical_points.fits")
    fits.HDUList([fits.PrimaryHDU(), fits.BinTableHDU.from_columns(coldefs, name="CRIT")]).writeto(outp, overwrite=True)
    return outp

print("saved:", save_critical_table())

### 3b · When do the critical points happen? Redshift / cosmic-time distributions

For each evolutionary anchor, compare the **redshift distribution** of fast vs slow
quenchers (with a matching **cosmic-time** axis on top). This shows *when* in cosmic
history each population reaches SF-peak, SFT, QT, etc. — e.g. whether slow quenchers
quench later (closer to z=0), consistent with retaining their reservoir longer.

In [ ]:
# redshift <-> cosmic-time mapping for the secondary axis (monotonic interpolation)
_zg = np.linspace(0, 8, 400)
_tg = COSMO.age(_zg).value                      # Gyr, decreasing in z
def _z_to_t(z): return np.interp(z, _zg, _tg)                 # z -> t [Gyr]
def _t_to_z(t): return np.interp(t, _tg[::-1], _zg[::-1])     # t -> z
def z_at_crit(stage):
    tt = np.array([r[f"t_{stage}"] for r in records])          # yr
    return np.interp(tt, t_cosmic_yr[::-1], redshift[::-1])

# common redshift binning across panels
_allz = np.concatenate([z_at_crit(st) for st in STAGES])
_zmax = np.nanpercentile(_allz[np.isfinite(_allz)], 98)
zbins = np.linspace(0, max(_zmax, 0.5), 16)

# rows = mass bin, cols = critical point; fast vs slow overlaid in each panel
ncol = len(STAGES)
fig, axes = plt.subplots(NBINS, ncol, figsize=(3.0 * ncol, 3.0 * NBINS), squeeze=False)
for bi in range(NBINS):
    binmask = mbin == bi
    for ai, st in enumerate(STAGES):
        ax = axes[bi, ai]
        zz = z_at_crit(st)
        for msk, c, lab in [(is_fast & binmask, "C3", "fast"), ((~is_fast) & binmask, "C0", "slow")]:
            v = zz[msk]; v = v[np.isfinite(v)]
            if len(v) == 0:
                continue
            ax.hist(v, bins=zbins, color=c, alpha=0.5, density=True, label=f"{lab} ({len(v)})")
            ax.axvline(np.median(v), color=c, ls="--", lw=1.2)
        if bi == 0:
            secax = ax.secondary_xaxis("top", functions=(_z_to_t, _t_to_z))
            secax.set_xlabel(f"{st}\nt [Gyr]", fontsize=8)
        if bi == NBINS - 1:
            ax.set_xlabel("redshift z")
        ax.legend(fontsize=6)
    axes[bi, 0].set_ylabel(f"logM* {MASS_BIN_LABELS[bi]}\npdf", fontsize=9)
fig.suptitle("Redshift (top: cosmic time) of each critical point — fast vs slow, by mass bin", y=1.01)
plt.tight_layout(); plt.savefig(os.path.join(PLOTDIR, "critical_point_redshift_dist.png"), dpi=130, bbox_inches="tight"); plt.show()

## 4 · Part 1 — catalogue distributions at the critical points

Using CAESAR integrated quantities only. At each evolutionary anchor we read each
galaxy's $M_{\rm dust}, M_{H_2}, M_{HI}, M_\star, M_{\rm gas}$ and both half-mass
radii from the history row that the anchor maps to. We then compare the
**distributions** between **fast** and **slow** quenchers, with the **SFT** anchor
playing the role of the star-forming reference state.

Key catalogue-level diagnostics of the hypothesis:
- $M_{\rm dust}/M_\star$, $M_{H_2}/M_\star$, $M_{HI}/M_\star$ — how much reservoir survives;
- $R_{\rm gas,1/2}/R_{\star,1/2}$ — is the cold gas **extended** vs centrally concentrated;
- dust-to-gas and HI/H₂ — phase of the surviving ISM.

In [ ]:
def gather_stage(prop, stage):
    '''Value of `prop` for every record at its `stage` anchor row (NaN if missing).'''
    rows = np.array([r[f"row_{stage}"] for r in records])
    out = np.full(len(records), np.nan)
    arr = P[prop]
    for i, (row, col) in enumerate(zip(rows, cols)):
        if row >= 0:
            out[i] = arr[row, col]
    return out

def ratio_stage(num, den, stage):
    a = gather_stage(num, stage); b = gather_stage(den, stage)
    return np.where(b > 0, a / b, np.nan)

# build a tidy dict: diagnostics[stage][quantity] = array over galaxies
DIAGS = {}
for st in STAGES:
    DIAGS[st] = {
        "Mdust":   gather_stage("masses.dust", st),
        "MH2":     gather_stage("masses.H2", st),
        "MHI":     gather_stage("masses.HI", st),
        "Mstar":   gather_stage("masses.stellar", st),
        "Mgas":    gather_stage("masses.gas", st),
        "dust/M*": ratio_stage("masses.dust", "masses.stellar", st),
        "H2/M*":   ratio_stage("masses.H2", "masses.stellar", st),
        "HI/M*":   ratio_stage("masses.HI", "masses.stellar", st),
        "dust/gas": ratio_stage("masses.dust", "masses.gas", st),
        "HI/H2":   ratio_stage("masses.HI", "masses.H2", st),
        "Rgas/Rstar": ratio_stage("radii.gas_half_mass", "radii.stellar_half_mass", st),
        "Rgas":    gather_stage("radii.gas_half_mass", st),
    }
    if "radii.dust_half_mass" in P:
        DIAGS[st]["Rdust/Rstar"] = ratio_stage("radii.dust_half_mass", "radii.stellar_half_mass", st)
        DIAGS[st]["Rdust"]       = gather_stage("radii.dust_half_mass", st)
HAS_RDUST = "radii.dust_half_mass" in P
print("diagnostics built for stages:", list(DIAGS.keys()), "| catalogue dust radius:", HAS_RDUST)

In [ ]:
# 4a. Distributions at each stage, fast vs slow — one column per mass bin
def violin_by_stage(qty, logy=True, ylabel=None):
    fig, axs = plt.subplots(1, NBINS, figsize=(5.0 * NBINS, 4.5), sharey=True)
    axs = np.atleast_1d(axs)
    for bi in range(NBINS):
        ax = axs[bi]; binmask = mbin == bi
        for j, st in enumerate(STAGES):
            for off, msk, c in [(-0.18, is_fast & binmask, "C3"), (0.18, (~is_fast) & binmask, "C0")]:
                v = DIAGS[st][qty][msk]
                v = v[np.isfinite(v) & (v > 0)] if logy else v[np.isfinite(v)]
                if len(v) < 3:
                    continue
                v = np.log10(v) if logy else v
                parts = ax.violinplot([v], positions=[j + off], widths=0.32, showmedians=True)
                for b in parts["bodies"]:
                    b.set_facecolor(c); b.set_alpha(0.6)
                for key in ("cmedians", "cbars", "cmins", "cmaxes"):
                    if key in parts:
                        parts[key].set_color(c)
        ax.set_xticks(range(len(STAGES))); ax.set_xticklabels(STAGES, rotation=20)
        ax.set_title(f"logM* {MASS_BIN_LABELS[bi]} (n={binmask.sum()})")
    axs[0].set_ylabel(ylabel or (f"log10 {qty}" if logy else qty))
    fig.suptitle(f"{qty} across stages  (red=fast, blue=slow; SFT = SF reference)", y=1.02)
    plt.tight_layout(); plt.savefig(os.path.join(PLOTDIR, f"dist_{qty.replace('/','_')}.png"), dpi=130, bbox_inches="tight"); plt.show()

for q in ["dust/M*", "H2/M*", "HI/M*"]:
    violin_by_stage(q)

In [ ]:
# 4b. Median stage evolution (fast vs slow) for the reservoir masses — one column per mass bin
def median_track(qty, logy=True):
    xs = np.arange(len(STAGES))
    fig, axs = plt.subplots(1, NBINS, figsize=(5.0 * NBINS, 4.5), sharey=True)
    axs = np.atleast_1d(axs)
    for bi in range(NBINS):
        ax = axs[bi]; binmask = mbin == bi
        for msk, c, lab in [(is_fast & binmask, "C3", "fast"), ((~is_fast) & binmask, "C0", "slow")]:
            med, lo, hi = [], [], []
            for st in STAGES:
                v = DIAGS[st][qty][msk]
                v = v[np.isfinite(v) & (v > 0)] if logy else v[np.isfinite(v)]
                if len(v) < 3:
                    med.append(np.nan); lo.append(np.nan); hi.append(np.nan); continue
                vv = np.log10(v) if logy else v
                med.append(np.median(vv)); lo.append(np.percentile(vv, 25)); hi.append(np.percentile(vv, 75))
            ax.plot(xs, med, "-o", color=c, label=lab)
            ax.fill_between(xs, lo, hi, color=c, alpha=0.15)
        ax.set_xticks(xs); ax.set_xticklabels(STAGES, rotation=20)
        ax.set_title(f"logM* {MASS_BIN_LABELS[bi]} (n={binmask.sum()})"); ax.legend(fontsize=8)
    axs[0].set_ylabel(f"median log10 {qty}" if logy else f"median {qty}")
    fig.suptitle(f"stage evolution of {qty}", y=1.02)
    plt.tight_layout(); plt.savefig(os.path.join(PLOTDIR, f"track_{qty.replace('/','_')}.png"), dpi=130, bbox_inches="tight"); plt.show()

for q in ["Mdust", "MH2", "MHI"]:
    median_track(q)

In [ ]:
# 4c. THE catalogue-level extension test: is the cold gas extended (R_gas/R_star)?
median_track("Rgas/Rstar", logy=False)
median_track("Rgas", logy=False)
# 4c-dust. Same, but for the DUST half-mass radius (needs radii.dust_half_mass in the
# history; rebuild with BUILD_HISTORY=True if HAS_RDUST is False).
if HAS_RDUST:
    median_track("Rdust/Rstar", logy=False)
    median_track("Rdust", logy=False)
else:
    print("radii.dust_half_mass not in history -> rebuild with BUILD_HISTORY=True to get dust-radius tracks.")
# 4d. surviving-ISM phase
median_track("dust/gas", logy=True)
median_track("HI/H2", logy=True)

### 4c-clock · Dust extension *around the moment of quenching*

The stage tracks above are sampled at discrete anchors. Here we show the catalogue dust
extension — $R_{\rm dust,1/2}/R_{\star,1/2}$ and $R_{\rm dust,1/2}$ — as a continuous
function of the **quenching clock**, stacking every galaxy's full history aligned to its
own QT (and, in the phase view, with the SFT→QT window normalised to [0,1]). If the
hypothesis holds, slow quenchers should keep an extended dust reservoir
($R_{\rm dust}/R_\star\gtrsim1$) through QT, while fast quenchers should see it shrink as
the gas is driven inward.

In [ ]:
# 4c-clock. Catalogue dust radius on the quenching clock, fast vs slow, per mass bin.
if HAS_RDUST:
    _Rd = P["radii.dust_half_mass"].astype(float)
    _Rs = P["radii.stellar_half_mass"].astype(float)
    with np.errstate(all="ignore"):
        _Rratio = np.where(_Rs > 0, _Rd / _Rs, np.nan)
    _grids = {"qt": np.linspace(-2.0, 2.0, 41), "phase": np.linspace(0.0, 1.5, 31)}
    _xlab  = {"qt": r"$t-t_{\rm QT}$ [Gyr]", "phase": r"$(t-t_{\rm SFT})/(t_{\rm QT}-t_{\rm SFT})$"}

    def _stack_cat(series2d, recs_sub, xgrid, mode, logy):
        rows = []
        for r in recs_sub:
            y = series2d[:, r["col"]]
            good = np.isfinite(y) & np.isfinite(t_cosmic_yr) & ((y > 0) if logy else np.isfinite(y))
            if good.sum() < 3:
                continue
            ts = t_cosmic_yr[good]; ys = np.log10(y[good]) if logy else y[good]
            o = np.argsort(ts); ts, ys = ts[o], ys[o]
            if mode == "qt":
                xx = (ts - r["t_qt"]) / 1e9
            else:
                den = r["t_qt"] - r["t_sft"]
                if den <= 0:
                    continue
                xx = (ts - r["t_sft"]) / den
            rows.append(np.interp(xgrid, xx, ys, left=np.nan, right=np.nan))
        if not rows:
            return None
        M = np.array(rows)
        with np.errstate(all="ignore"):
            return np.nanmedian(M, axis=0), np.nanpercentile(M, 25, axis=0), np.nanpercentile(M, 75, axis=0)

    for qname, series2d, logy, ylab in [
            ("Rdust_over_Rstar", _Rratio, False, r"$R_{\rm dust,1/2}/R_{\star,1/2}$"),
            ("Rdust",            _Rd,     True,  r"$\log_{10} R_{\rm dust,1/2}$ [kpc/h]")]:
        fig, axes = plt.subplots(NBINS, 2, figsize=(11, 3.6 * NBINS), squeeze=False)
        for bi in range(NBINS):
            binmask = mbin == bi
            frecs = [r for r, f, b in zip(records, is_fast, binmask) if (f and b)]
            srecs = [r for r, f, b in zip(records, is_fast, binmask) if ((not f) and b)]
            for mj, mode in enumerate(["qt", "phase"]):
                ax = axes[bi, mj]; xg = _grids[mode]
                for recs, c, name in [(frecs, "C3", "fast"), (srecs, "C0", "slow")]:
                    res = _stack_cat(series2d, recs, xg, mode, logy)
                    if res is None:
                        continue
                    med, lo, hi = res
                    ax.plot(xg, med, "-", color=c, lw=2, label=f"{name} (n={len(recs)})")
                    ax.fill_between(xg, lo, hi, color=c, alpha=0.15)
                ax.axvline(0, ls=":", c="k", lw=1)
                if mode == "phase":
                    ax.axvline(1, ls="--", c="0.5", lw=1)
                if not logy:
                    ax.axhline(1, ls=":", c="0.6", lw=0.8)      # dust as extended as stars
                if bi == NBINS - 1:
                    ax.set_xlabel(_xlab[mode])
                if mj == 0:
                    ax.set_ylabel(f"logM* {MASS_BIN_LABELS[bi]}\n{ylab}", fontsize=9)
                if bi == 0:
                    ax.set_title("QT-aligned" if mode == "qt" else "phase-norm")
                ax.legend(fontsize=7)
        fig.suptitle(f"{ylab} around quenching — fast vs slow, by mass bin", y=1.005)
        plt.tight_layout(); plt.savefig(os.path.join(PLOTDIR, f"clock_{qname}.png"), dpi=130, bbox_inches="tight"); plt.show()
else:
    print("radii.dust_half_mass not in history -> rebuild with BUILD_HISTORY=True for the dust-radius clock.")

In [ ]:
# 4e. Example sSFR tracks with critical points marked — rows = mass bin, cols = class
def show_examples(n_each=2):
    fig, axes = plt.subplots(NBINS, 2, figsize=(13, 4.0 * NBINS), sharex=True)
    axes = np.atleast_2d(axes)
    for bi in range(NBINS):
        binmask = mbin == bi
        for col_i, (cmask, title) in enumerate([(is_fast & binmask, "fast"), ((~is_fast) & binmask, "slow")]):
            ax = axes[bi, col_i]
            for i in np.where(cmask)[0][:n_each]:
                r = records[i]; col = r["col"]
                mstar = P["masses.stellar"][:, col]; sfr = P["sfr"][:, col]
                ssfr = np.where(mstar > 0, sfr / mstar, np.nan)
                ax.plot(t_cosmic_yr / 1e9, ssfr, "-", lw=1, alpha=0.8, label=f"gid {r['gid']}")
                for st, mk in [("sft", "^"), ("qt", "v"), ("dust_peak", "*")]:
                    row = r[f"row_{st}"]
                    if row >= 0:
                        ax.scatter(t_cosmic_yr[row] / 1e9, ssfr[row], marker=mk, s=70, zorder=5)
            ax.set_yscale("log"); ax.set_title(f"{title} — logM* {MASS_BIN_LABELS[bi]}"); ax.legend(fontsize=7)
            if bi == NBINS - 1:
                ax.set_xlabel("cosmic time [Gyr]")
        axes[bi, 0].set_ylabel("sSFR [1/yr]")
    plt.tight_layout(); plt.show()
show_examples()

## 4b · AGN-feeding cross-check (BH accretion / AGN energy)

The dust-survival hypothesis hinges on a *negative*: the gas **did not** feed the
AGN, so there was no central energy injection to destroy the grains. We test that
link directly with the black hole's own history. CAESAR exposes galaxy-level BH mass
(`masses.bh`), accretion rate (`bhmdot`) and Eddington ratio (`bh_fedd`); SIMBA's
**jet-mode** (the channel that actually heats/ejects gas) triggers at
$M_{\rm BH} > 10^{7.5}\,M_\odot$ **and** $f_{\rm Edd} < 0.2$.

For each galaxy we read these along the progenitor chain (same index machinery as the
property history) and, over the **SFT→QT** window, compute:
- the BH mass *grown* $\Delta M_{\rm BH}$ and the integrated feedback energy
  $E_{\rm AGN}=\epsilon_r\,c^2\!\int \dot M_{\rm BH}\,dt$ (with $\epsilon_r=0.1$);
- the **fraction of time spent in jet mode**.

**Prediction:** dust-rich / slow quenchers should show **little AGN feeding** around
quenching (low $E_{\rm AGN}$, little jet-mode time), and residual dust at z=0 should
**anti-correlate** with $E_{\rm AGN}$ / jet-mode time.

> *Units caveat:* `bhmdot` is taken as $M_\odot\,{\rm yr}^{-1}$ and `masses.bh` as
> $M_\odot$ (CAESAR convention). Confirm against your build; the §0.1 probe lists the
> available keys and `BH_CANDIDATES` below tries the common name variants.

In [ ]:
BH_HIST_PATH = os.path.join(SFHDIR, "bh_history_passive_z0.hdf5")
EPS_R = 0.1   # radiative/feedback efficiency for the AGN energy proxy

BH_CANDIDATES = {
    "bh_mass": ["masses.bh", "masses.bh_mass", "bhmass"],
    "bh_mdot": ["bhmdot", "bh_mdot"],
    "bh_fedd": ["bh_fedd", "bhfedd", "fedd"],
}

def _resolve_bh_path(f, cands):
    for c in cands:
        for p in (f"galaxy_data/dicts/{c}", f"galaxy_data/{c}"):
            if p in f:
                return p
    return None

if BUILD_BH:
    cs0 = sim.load_catalog(snap=START_SNAP)
    _hb = HDF5BuildHistory(sim, cs0, progfilename=PROG_FILE)
    _hb.get_history_indx(galaxy_ids, START_SNAP, END_SNAP)
    pidx = np.vstack([_hb.history_indx[str(s)] for s in snaps_arr])   # (n_snap, n_gal)
    del cs0; gc.collect()

    BH = {k: np.full((n_snap, n_gal), np.nan) for k in BH_CANDIDATES}
    resolved = {}
    bh_skipped = []
    for ri, snap in enumerate(snaps_arr):
        snap = int(snap)
        if snap in CORRUPT_SNAPS:
            bh_skipped.append(snap); continue        # leave NaN for this row
        try:
            with h5py.File(sim.get_caesar_file(snap), "r") as f:
                valid = np.isfinite(pidx[ri])
                cols_valid = np.where(valid)[0]
                vi = pidx[ri][valid].astype(int)
                for k, cands in BH_CANDIDATES.items():
                    p = _resolve_bh_path(f, cands)
                    resolved[k] = p
                    if p is not None:
                        BH[k][ri, cols_valid] = f[p][:][vi]
        except (OSError, KeyError) as e:
            # truncated/corrupt catalog -> record it, leave NaN row, keep going
            print(f"  [skip] snap {snap}: {type(e).__name__}: {e}")
            CORRUPT_SNAPS.add(snap); bh_skipped.append(snap); continue
    with h5py.File(BH_HIST_PATH, "w") as f:
        for k, arr in BH.items():
            f.create_dataset(k, data=arr)
    print("BH history written:", BH_HIST_PATH)
    print("resolved keys:", resolved)
    if bh_skipped:
        print(f"skipped {len(bh_skipped)} corrupt snapshot(s): {sorted(set(bh_skipped))}")
else:
    print("BUILD_BH=False -> expecting", BH_HIST_PATH)

In [ ]:
# load BH history and derive AGN-feeding metrics per galaxy (aligned to `records`)
from astropy import units as u
from astropy import constants as const

with h5py.File(BH_HIST_PATH, "r") as f:
    BH = {k: f[k][:] for k in f.keys()}

def bh_stage(arr, stage):
    rows = np.array([r[f"row_{stage}"] for r in records])
    out = np.full(len(records), np.nan)
    for i, (row, col) in enumerate(zip(rows, cols)):
        if row >= 0:
            out[i] = arr[row, col]
    return out

def _sft_qt_rows(r):
    rs, rq = r["row_sft"], r["row_qt"]
    if rs < 0 or rq < 0:
        return np.array([], dtype=int)
    lo, hi = sorted((rs, rq))
    return np.arange(lo, hi + 1)

c2_erg_per_Msun = (const.c ** 2 * u.Msun).to("erg").value     # E = M c^2 for 1 Msun

dMbh, Eagn, jetfrac, mdot_qt = (np.full(len(records), np.nan) for _ in range(4))
for i, r in enumerate(records):
    col = r["col"]
    mdot_qt[i] = BH["bh_mdot"][r["row_qt"], col] if r["row_qt"] >= 0 else np.nan
    idx = _sft_qt_rows(r)
    if len(idx) < 2:
        continue
    t = t_cosmic_yr[idx]; o = np.argsort(t); idx = idx[o]; t = t[o]
    md = BH["bh_mdot"][idx, col]; mb = BH["bh_mass"][idx, col]; fe = BH["bh_fedd"][idx, col]
    good = np.isfinite(md) & np.isfinite(t)
    if good.sum() >= 2:
        M_acc = np.trapz(md[good], t[good])               # Msun (Msun/yr * yr)
        Eagn[i] = EPS_R * M_acc * c2_erg_per_Msun          # erg
    if np.isfinite(mb).sum() >= 2:
        mbv = mb[np.isfinite(mb)]
        dMbh[i] = mbv[-1] - mbv[0]
    jet = np.isfinite(mb) & np.isfinite(fe) & (mb > 10 ** 7.5) & (fe < 0.2)
    valid_state = np.isfinite(mb) & np.isfinite(fe)
    jetfrac[i] = jet.sum() / valid_state.sum() if valid_state.sum() > 0 else np.nan

print(f"galaxies with E_AGN(SFT->QT): {np.isfinite(Eagn).sum()} / {len(records)}")

In [ ]:
# 4b-i. median BH accretion rate & Eddington ratio across stages — one column per mass bin
def bh_median_track(arr, label, logy=True):
    xs = np.arange(len(STAGES))
    fig, axs = plt.subplots(1, NBINS, figsize=(5.0 * NBINS, 4.5), sharey=True)
    axs = np.atleast_1d(axs)
    for bi in range(NBINS):
        ax = axs[bi]; binmask = mbin == bi
        for msk, c, lab in [(is_fast & binmask, "C3", "fast"), ((~is_fast) & binmask, "C0", "slow")]:
            med, lo, hi = [], [], []
            for st in STAGES:
                v = bh_stage(arr, st)[msk]
                v = v[np.isfinite(v) & (v > 0)] if logy else v[np.isfinite(v)]
                if len(v) < 3:
                    med.append(np.nan); lo.append(np.nan); hi.append(np.nan); continue
                vv = np.log10(v) if logy else v
                med.append(np.median(vv)); lo.append(np.percentile(vv, 25)); hi.append(np.percentile(vv, 75))
            ax.plot(xs, med, "-o", color=c, label=lab); ax.fill_between(xs, lo, hi, color=c, alpha=0.15)
        ax.set_xticks(xs); ax.set_xticklabels(STAGES, rotation=20)
        ax.set_title(f"logM* {MASS_BIN_LABELS[bi]} (n={binmask.sum()})"); ax.legend(fontsize=8)
    axs[0].set_ylabel(("median log10 " if logy else "median ") + label)
    fig.suptitle(f"stage evolution of {label}", y=1.02)
    plt.tight_layout(); plt.savefig(os.path.join(PLOTDIR, f"bh_{label}.png".replace(" ", "_")), dpi=130, bbox_inches="tight"); plt.show()

bh_median_track(BH["bh_mdot"], "BH Mdot [Msun/yr]", logy=True)
bh_median_track(BH["bh_fedd"], "f_Edd", logy=True)

In [ ]:
# 4b-ii. THE cross-check: residual dust at z=0 vs AGN feeding during SFT->QT
# rows = mass bin, cols = the three AGN-feeding metrics; Spearman printed per bin.
from scipy.stats import spearmanr
dust_z0 = DIAGS["z0"]["dust/M*"]          # aligned to `records`
metrics = [("E_AGN", Eagn, r"$\log_{10} E_{\rm AGN}$(SFT$\to$QT) [erg]", True),
           ("dMbh",  dMbh, r"$\log_{10}\,\Delta M_{\rm BH}$(SFT$\to$QT) [$M_\odot$]", True),
           ("jetfrac", jetfrac, "jet-mode time fraction (SFT$\to$QT)", False)]
fig, axes = plt.subplots(NBINS, 3, figsize=(15, 4.2 * NBINS), squeeze=False)
for bi in range(NBINS):
    binmask = mbin == bi
    for mi, (name, x, xlab, logx) in enumerate(metrics):
        ax = axes[bi, mi]
        for cmask, c, lab in [(is_fast & binmask, "C3", "fast"), ((~is_fast) & binmask, "C0", "slow")]:
            xv = np.log10(x[cmask]) if logx else x[cmask]
            ax.scatter(xv, np.log10(dust_z0[cmask]), s=18, c=c, label=lab, alpha=0.8)
        if bi == NBINS - 1:
            ax.set_xlabel(xlab)
        ax.set_ylabel(r"$\log_{10}(M_{\rm dust}/M_\star)_{z=0}$"); ax.legend(fontsize=7)
        m = binmask & np.isfinite(x) & np.isfinite(dust_z0) & (dust_z0 > 0)
        if m.sum() > 5:
            rho, p = spearmanr(x[m], dust_z0[m])
            ax.set_title(f"logM* {MASS_BIN_LABELS[bi]}: rho={rho:+.2f} (p={p:.2g}, N={m.sum()})", fontsize=8)
            print(f"[{MASS_BIN_LABELS[bi]:9s}] Spearman(dust z=0, {name:7s}): rho={rho:+.2f} p={p:.3g} N={m.sum()}")
        else:
            ax.set_title(f"logM* {MASS_BIN_LABELS[bi]}: N<6", fontsize=8)
fig.suptitle("residual dust vs AGN feeding, by mass bin  (hypothesis: anti-correlation)", y=1.005)
plt.tight_layout(); plt.savefig(os.path.join(PLOTDIR, "agn_feeding_vs_residual_dust.png"), dpi=130, bbox_inches="tight"); plt.show()

### 4b-iii · Full BH accretion / Eddington-ratio history — are fast & slow degenerate?

The stage-sampled view (§4b-i) only sees the anchors. Here we plot the **entire**
$\dot M_{\rm BH}(z)$ and $f_{\rm Edd}(z)$ histories — median with 25–75% band over all
fast vs all slow quenchers, vs **redshift** with a **cosmic-time** top axis — to see
whether the two populations actually trace *different* BH-growth paths or whether the
accretion history is **degenerate** between them. A KS test on the pooled per-snapshot
values and on the total integrated $E_{\rm AGN}$ quantifies any difference.

In [ ]:
from scipy.stats import ks_2samp

def _med_band(arr2d, colsel, positive=True):
    sub = arr2d[:, colsel].astype(float)
    if positive:
        sub = np.where(sub > 0, sub, np.nan)
    with np.errstate(all="ignore"):
        return (np.nanmedian(sub, axis=1),
                np.nanpercentile(sub, 25, axis=1),
                np.nanpercentile(sub, 75, axis=1))

def _pooled(arr2d, colsel):
    v = arr2d[:, colsel].ravel()
    return v[np.isfinite(v) & (v > 0)]

def _total_Eagn(col):
    t = t_cosmic_yr[::-1]; md = BH['bh_mdot'][::-1, col]
    good = np.isfinite(md) & (md >= 0)
    if good.sum() < 2:
        return np.nan
    return EPS_R * np.trapz(md[good], t[good]) * c2_erg_per_Msun

# rows = [bh_mdot, f_Edd], cols = mass bins; fast vs slow median+IQR vs redshift
keys = [("bh_mdot", r"$\dot M_{\rm BH}$ [$M_\odot$ yr$^{-1}$]"), ("bh_fedd", r"$f_{\rm Edd}$")]
fig, axes = plt.subplots(2, NBINS, figsize=(5.0 * NBINS, 9), squeeze=False)
for ki, (key, lab) in enumerate(keys):
    for bi in range(NBINS):
        ax = axes[ki, bi]; binmask = mbin == bi
        for cls_mask, c, name in [(is_fast & binmask, "C3", "fast"), ((~is_fast) & binmask, "C0", "slow")]:
            colsel = cols[cls_mask]
            if len(colsel) == 0:
                continue
            med, lo, hi = _med_band(BH[key], colsel)
            ax.plot(redshift, med, "-", color=c, lw=2, label=f"{name} (n={len(colsel)})")
            ax.fill_between(redshift, lo, hi, color=c, alpha=0.15)
        ax.set_yscale("log"); ax.invert_xaxis(); ax.legend(fontsize=7)
        if ki == 0:
            ax.set_title(f"logM* {MASS_BIN_LABELS[bi]}")
            secax = ax.secondary_xaxis("top", functions=(_z_to_t, _t_to_z)); secax.set_xlabel("t [Gyr]", fontsize=8)
        if ki == 1:
            ax.set_xlabel("redshift z")
    axes[ki, 0].set_ylabel("median " + lab)
fig.suptitle("BH accretion / Eddington history — fast vs slow, by mass bin", y=1.01)
plt.tight_layout(); plt.savefig(os.path.join(PLOTDIR, "bh_accretion_history_fast_slow.png"), dpi=130, bbox_inches="tight"); plt.show()

# ── quantify degeneracy, per mass bin ──────────────────────────────────────
for bi in range(NBINS):
    binmask = mbin == bi
    fast_cols = cols[is_fast & binmask]; slow_cols = cols[(~is_fast) & binmask]
    print(f"--- logM* {MASS_BIN_LABELS[bi]} (fast {len(fast_cols)}, slow {len(slow_cols)}) ---")
    for key, lab in [("bh_mdot", "Mdot"), ("bh_fedd", "f_Edd")]:
        a = np.log10(_pooled(BH[key], fast_cols)); b = np.log10(_pooled(BH[key], slow_cols))
        if len(a) > 5 and len(b) > 5:
            ks = ks_2samp(a, b)
            print(f"  KS pooled log {lab:6s}: D={ks.statistic:.3f} p={ks.pvalue:.2g} -> "
                  f"{'DEGENERATE' if ks.pvalue > 0.05 else 'distinct'}")
    E_fast = np.array([_total_Eagn(c) for c in fast_cols]); E_fast = E_fast[np.isfinite(E_fast) & (E_fast > 0)]
    E_slow = np.array([_total_Eagn(c) for c in slow_cols]); E_slow = E_slow[np.isfinite(E_slow) & (E_slow > 0)]
    if len(E_fast) > 5 and len(E_slow) > 5:
        ks = ks_2samp(np.log10(E_fast), np.log10(E_slow))
        print(f"  KS total log E_AGN : D={ks.statistic:.3f} p={ks.pvalue:.2g} -> "
              f"{'DEGENERATE' if ks.pvalue > 0.05 else 'distinct'}")

### 4b-iv · Quenching-clock view — does a difference appear *around* quenching?

If the full history (4b-iii) is degenerate, a difference may still hide in the brief
window when quenching actually happens. We re-stack every galaxy's BH history on its
**own quenching clock**, two ways:

- **QT-aligned:** $x = t - t_{QT}$ [Gyr] — absolute lookback around the quenching event;
- **phase-normalized:** $\phi = (t-t_{SFT})/(t_{QT}-t_{SFT})$, so $\phi{=}0$ is SFT, $\phi{=}1$
  is QT — this removes the $\tau_q$ duration difference and compares fast vs slow *by
  phase*, isolating any contrast in how hard the BH is fed approaching quenching.

KS tests on the **peak** $\dot M_{\rm BH}$ and $f_{\rm Edd}$ *within* each galaxy's
SFT→QT window quantify it.

In [ ]:
# stack a BH quantity on the quenching clock and return median + 25/75 band
def _stack(recs_sub, key, xgrid, mode):
    rows_acc = []
    for r in recs_sub:
        y = BH[key][:, r["col"]].astype(float)
        good = np.isfinite(y) & (y > 0) & np.isfinite(t_cosmic_yr)
        if good.sum() < 3:
            continue
        ts = t_cosmic_yr[good]; ys = np.log10(y[good])
        o = np.argsort(ts); ts, ys = ts[o], ys[o]
        if mode == "qt":
            xx = (ts - r["t_qt"]) / 1e9
        else:
            den = r["t_qt"] - r["t_sft"]
            if den <= 0:
                continue
            xx = (ts - r["t_sft"]) / den
        rows_acc.append(np.interp(xgrid, xx, ys, left=np.nan, right=np.nan))
    if not rows_acc:
        return None
    M = np.array(rows_acc)
    with np.errstate(all="ignore"):
        return np.nanmedian(M, axis=0), np.nanpercentile(M, 25, axis=0), np.nanpercentile(M, 75, axis=0)

grids   = {"qt": np.linspace(-2.0, 2.0, 41), "phase": np.linspace(0.0, 1.5, 31)}
xlabels = {"qt": r"$t - t_{\rm QT}$ [Gyr]",
           "phase": r"phase $(t-t_{\rm SFT})/(t_{\rm QT}-t_{\rm SFT})$"}
qkeys = [("bh_mdot", r"$\log\,\dot M_{\rm BH}$"), ("bh_fedd", r"$\log f_{\rm Edd}$")]

def _window_peak(key, sel):
    fa, sa = [], []
    for i, (r, isf) in enumerate(zip(records, is_fast)):
        if not sel[i]:
            continue
        idx = _sft_qt_rows(r)
        if len(idx) < 2:
            continue
        v = BH[key][idx, r["col"]]; v = v[np.isfinite(v) & (v > 0)]
        if len(v) == 0:
            continue
        (fa if isf else sa).append(np.max(v))
    return np.array(fa), np.array(sa)

# one quenching-clock figure (2 qty x 2 clock modes) per mass bin
for bi in range(NBINS):
    binmask = mbin == bi
    fast_recs = [r for r, f, b in zip(records, is_fast, binmask) if (f and b)]
    slow_recs = [r for r, f, b in zip(records, is_fast, binmask) if ((not f) and b)]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
    for ci, (key, lab) in enumerate(qkeys):
        for ri, mode in enumerate(["qt", "phase"]):
            ax = axes[ri, ci]; xg = grids[mode]
            for recs, c, name in [(fast_recs, "C3", "fast"), (slow_recs, "C0", "slow")]:
                res = _stack(recs, key, xg, mode)
                if res is None:
                    continue
                med, lo, hi = res
                ax.plot(xg, med, "-", color=c, lw=2, label=f"{name} (n={len(recs)})")
                ax.fill_between(xg, lo, hi, color=c, alpha=0.15)
            ax.axvline(0, ls=":", c="k", lw=1)
            if mode == "phase":
                ax.axvline(1, ls="--", c="0.5", lw=1)
            ax.set_xlabel(xlabels[mode]); ax.set_ylabel(lab); ax.legend(fontsize=7)
            ax.set_title(f"{lab} — {'QT-aligned' if mode == 'qt' else 'phase-norm'}")
    fig.suptitle(f"Quenching clock — logM* {MASS_BIN_LABELS[bi]}", y=1.01)
    plt.tight_layout(); plt.savefig(os.path.join(PLOTDIR, f"bh_quenching_clock_bin{bi}.png"), dpi=130, bbox_inches="tight"); plt.show()

    print(f"--- logM* {MASS_BIN_LABELS[bi]} ---")
    for key, lab in [("bh_mdot", "peak Mdot (SFT->QT)"), ("bh_fedd", "peak f_Edd (SFT->QT)")]:
        fa, sa = _window_peak(key, binmask)
        if len(fa) > 5 and len(sa) > 5:
            ks = ks_2samp(np.log10(fa), np.log10(sa))
            print(f"  KS {lab:22s}: D={ks.statistic:.3f} p={ks.pvalue:.2g} -> "
                  f"{'DEGENERATE' if ks.pvalue > 0.05 else 'distinct'}")

## 5 · Part 2 — particle-level radial profiles & dust extension

The catalogue gives integrated masses and a single half-mass radius. To test
"**equilibrium at large radii**" properly we need the **radial distribution** of
dust, H₂ and HI. For a manageable subset of fast and slow quenchers we extract the
member particles at each critical point, build **face-on surface-density profiles**,
and measure the **effective radii** $R_{50}, R_{90}$ of each component — then ask
whether the **dust extension changes with time** differently for fast vs slow quenchers.

> Extraction reuses `simbanator.analysis.extract_particles`; profiles are built from
> the resulting per-galaxy HDF5. Field names for the dust/HI/H₂ split depend on the
> snapshot build — the discovery helper below prints what is present so the mapping
> in `gas_component_masses` can be confirmed/adjusted on the cluster.
>
> **Selection logic.** Candidates are ranked by z=0 dust mass; a galaxy is accepted as
> soon as it yields a valid dust profile at every stage in `REQUIRED_STAGES` (default
> `z0` + `post_quench`). The other stages (`sft`, `qt`) are best-effort and simply drawn
> when available — they sit at high redshift where progenitor chains often break and
> early dust is sparse, so demanding *all four* would discard almost every galaxy. The
> §5a output prints a drop-out **funnel** (`stage:reason -> count`) so you can see exactly
> where galaxies are lost and decide whether to relax `REQUIRED_STAGES`.

In [ ]:
# 5·pre — progenitor-index matrix aligned to the history (cheap: reads the progenitor
# FITS + one group catalogue for the box size). Translates a (history row, galaxy
# column) into the CAESAR galaxy index at that snapshot, for any galaxy/stage.
cs0 = sim.load_catalog(snap=START_SNAP)
_hP = HDF5BuildHistory(sim, cs0, progfilename=PROG_FILE)
_hP.get_history_indx(galaxy_ids, START_SNAP, END_SNAP)
_PROG_INDEX = np.vstack([_hP.history_indx[str(s)] for s in snaps_arr])   # (n_snap, n_gal)
del cs0; gc.collect()
print("_PROG_INDEX:", _PROG_INDEX.shape)

In [ ]:
# particle-work configuration. Candidates are RANKED by z=0 dust mass (we want dust to
# profile). A galaxy is ACCEPTED once it has a valid dust profile at every stage in
# REQUIRED_STAGES; the other PART_STAGES are best-effort (drawn if available). The old
# "all four stages required" rule discarded almost everything, because SFT/QT sit at
# high z where progenitor chains break and early dust is often absent.
N_PER_CLASS = 10       # galaxies wanted per class
MAX_TRY     = 25000    # max candidates to attempt per class before giving up
PART_STAGES = ["sft", "qt", "post_quench", "z0"]   # anchors to profile (best-effort)
REQUIRED_STAGES = ["z0", "post_quench"]            # must have a dust profile here to accept
PART_DIR    = os.path.join(OUT, "filtered_particles")
PART_MBIN   = None     # None = all mass bins; or set 0/1/2 to restrict the particle work
                       # to one bin (per-class-per-bin particle counts are usually too
                       # sparse to facet the heavy profiles, so default pools all bins).

rank_key = P["masses.dust"][0, cols]   # rank by z=0 dust mass (NaN/0 sink to the bottom)
rank_key = np.where(np.isfinite(rank_key), rank_key, -np.inf)
_inbin = np.ones(len(records), bool) if PART_MBIN is None else (mbin == PART_MBIN)
_fi = np.where(is_fast & _inbin)[0];    fast_cands = list(_fi[np.argsort(-rank_key[_fi])])
_si = np.where((~is_fast) & _inbin)[0]; slow_cands = list(_si[np.argsort(-rank_key[_si])])
print(f"total fast={int(is_fast.sum())}, slow={int((~is_fast).sum())}; "
      f"mass-bin filter PART_MBIN={PART_MBIN}")
print(f"candidates after bin filter: {len(fast_cands)} fast, {len(slow_cands)} slow "
      f"(want {N_PER_CLASS} each; required stages={REQUIRED_STAGES})")

In [ ]:
# 5·helpers — file lookup, unit conversions, and per-particle dust/HI/H2 masses.
def find_extracted_file(klass, snap, gidx):
    d = os.path.join(PART_DIR, f"snap_{snap:03d}")
    if not os.path.isdir(d):
        return None
    cand = [f for f in os.listdir(d) if f.endswith(".h5") and f"gal{gidx:06d}" in f and klass in f]
    return os.path.join(d, cand[0]) if cand else None

def header_units(f):
    h = f["Header"].attrs if "Header" in f else {}
    return float(h.get("Time", 1.0)), float(h.get("HubbleParam", 0.68))   # (scale factor, h)

def to_kpc(coords_code, a, hub):
    return coords_code * a / hub          # comoving kpc/h -> physical kpc

def msun(mass_code, hub):
    return mass_code * 1e10 / hub         # 1e10 Msun/h -> Msun

def gas_component_masses(g, hub):
    '''Per-gas-particle (m_gas, m_dust, m_HI, m_H2) in Msun; NaN where a field is absent.'''
    m_gas = msun(g["Masses"][:], hub)
    m_dust = np.full_like(m_gas, np.nan)
    for key in ("Dust_Masses", "DustMasses"):
        if key in g:
            m_dust = msun(g[key][:], hub); break
    if "Metallicity" in g and g["Metallicity"].ndim == 2 and g["Metallicity"].shape[1] >= 2:
        XH = np.clip(1.0 - g["Metallicity"][:, 0] - g["Metallicity"][:, 1], 0.0, 1.0)
    else:
        XH = np.full_like(m_gas, 0.76)
    m_H = m_gas * XH
    fneut = g["NeutralHydrogenAbundance"][:] if "NeutralHydrogenAbundance" in g else np.full_like(m_gas, np.nan)
    fmol = None
    for key in ("FractionH2", "fH2", "GrackleH2", "f_H2"):
        if key in g:
            fmol = g[key][:]; break
    if fmol is not None:
        m_H2 = m_H * fneut * fmol; m_HI = m_H * fneut * (1.0 - fmol)
    else:
        m_H2 = np.full_like(m_gas, np.nan); m_HI = m_H * fneut
    return m_gas, m_dust, m_HI, m_H2

In [ ]:
# 5d. face-on surface-density profiles + effective radii
def sigma_star(Rs, smass, edges, area):
    s, _ = np.histogram(Rs, bins=edges, weights=smass)
    return s / area

def radial_profiles(path, rmax_kpc=50.0, nbins=25):
    with h5py.File(path, "r") as f:
        a, hub = header_units(f)
        g = f["PartType0"]
        pos = to_kpc(g["Coordinates"][:], a, hub)
        m_gas, m_dust, m_HI, m_H2 = gas_component_masses(g, hub)
        # stellar mass/positions for the orientation frame + R50(stars)
        if "PartType4" in f:
            spos = to_kpc(f["PartType4"]["Coordinates"][:], a, hub)
            smass = msun(f["PartType4"]["Masses"][:], hub)
        else:
            spos, smass = pos, m_gas

    center = shrink_center(spos, masses=smass)
    _, _, evecs, _ = principal_axes(spos - center, masses=smass)
    R = np.sqrt(np.sum(rotate_to_frame(pos, center, evecs)[:, :2] ** 2, axis=1))   # face-on cyl. radius
    Rs = np.sqrt(np.sum(rotate_to_frame(spos, center, evecs)[:, :2] ** 2, axis=1))

    edges = np.linspace(0, rmax_kpc, nbins + 1)
    rmid = 0.5 * (edges[:-1] + edges[1:])
    area = np.pi * (edges[1:] ** 2 - edges[:-1] ** 2)        # kpc^2

    def sigma(mass):
        if not np.any(np.isfinite(mass)):
            return np.full(nbins, np.nan)
        s, _ = np.histogram(R, bins=edges, weights=np.nan_to_num(mass))
        return s / area

    def r_enc(mass, frac):
        m = np.nan_to_num(mass)
        o = np.argsort(R); c = np.cumsum(m[o])
        if c[-1] <= 0:
            return np.nan
        return float(R[o][np.searchsorted(c, frac * c[-1])])

    prof = {
        "rmid": rmid,
        "sigma_dust": sigma(m_dust), "sigma_HI": sigma(m_HI),
        "sigma_H2": sigma(m_H2), "sigma_star": sigma_star(Rs, smass, edges, area),
        "R50_dust": r_enc(m_dust, 0.5), "R90_dust": r_enc(m_dust, 0.9),
        "R50_HI": r_enc(m_HI, 0.5), "R50_H2": r_enc(m_H2, 0.5),
        "R50_star": float(np.sort(Rs)[np.searchsorted(np.cumsum(smass[np.argsort(Rs)]), 0.5*np.sum(smass))]) if np.sum(smass) > 0 else np.nan,
    }
    return prof

In [ ]:
# 5a. Build dust radial profiles. A galaxy is accepted once it has a valid dust profile
# at every stage in REQUIRED_STAGES; remaining PART_STAGES are best-effort. Each stage is
# handled independently, so a single broken-chain / corrupt-snapshot / dustless stage no
# longer discards the whole galaxy. FAIL tallies *why* stages drop out (the funnel).
from collections import Counter
_cat_cache = {}
FAIL = Counter()

def _stage_plan(rec):
    # one (stage, snap, gidx) per stage that has a tracked progenitor; invalid stages
    # are skipped (counted), not fatal to the galaxy.
    plan = []
    for st in PART_STAGES:
        row = rec[f"row_{st}"]
        if row < 0:
            FAIL[f"{st}:no_row"] += 1; continue
        gidx = _PROG_INDEX[row, rec["col"]]
        if not np.isfinite(gidx):
            FAIL[f"{st}:broken_chain"] += 1; continue
        plan.append((st, int(snaps_arr[row]), int(gidx)))
    return plan

def _ensure_extracted(klass, snap, gidx):
    path = find_extracted_file(klass, snap, gidx)
    if path and os.path.exists(path):
        return path                         # already on disk -> don't touch the snapshot
    if not RUN_PARTICLES:
        return None
    if snap not in _cat_cache:
        _cat_cache[snap] = sim.load_catalog(snap=snap)
    extract_particles(cs=_cat_cache[snap], simfile=sim.get_snapshot_file(snap), snap=snap,
                      galaxy_id=gidx, sim_name=SIM_NAME, prefix=f"resdust_{klass}",
                      ptypes=("PartType0", "PartType4"), overwrite=False, verbose=0)
    return find_extracted_file(klass, snap, gidx)

def _profile_stage(klass, st, snap, gidx):
    if snap in CORRUPT_SNAPS:
        FAIL[f"{st}:corrupt"] += 1; return None
    try:
        path = _ensure_extracted(klass, snap, gidx)
    except OSError:
        CORRUPT_SNAPS.add(snap); FAIL[f"{st}:corrupt"] += 1; return None
    except Exception:
        FAIL[f"{st}:extract_err"] += 1; return None
    if not path or not os.path.exists(path):
        FAIL[f"{st}:no_file"] += 1; return None       # e.g. galaxy had no extractable particles
    try:
        p = radial_profiles(path)
    except Exception:
        FAIL[f"{st}:profile_err"] += 1; return None
    if not np.isfinite(p.get("R50_dust", np.nan)):
        FAIL[f"{st}:no_dust"] += 1; return None        # no usable dust radial info
    return p

def _try_candidate(rec, klass):
    profs = {}
    for st, snap, gidx in _stage_plan(rec):
        p = _profile_stage(klass, st, snap, gidx)
        if p is not None:
            profs[st] = p
    if all(s in profs for s in REQUIRED_STAGES):
        return profs
    FAIL["rejected:missing_required"] += 1
    return None

def _collect(cands, klass, n_want):
    accepted, profiles, tried = [], {}, 0
    for ci in cands:
        if tried >= MAX_TRY or len(accepted) >= n_want:
            break
        tried += 1
        profs = _try_candidate(records[ci], klass)
        if profs is None:
            continue
        accepted.append(records[ci])
        for st, p in profs.items():
            profiles[(records[ci]["gid"], st)] = p
    return accepted, profiles, tried

if RUN_PARTICLES or os.path.isdir(PART_DIR):
    fast_acc, _fp, _ft = _collect(fast_cands, "fast", N_PER_CLASS)
    slow_acc, _sp, _st = _collect(slow_cands, "slow", N_PER_CLASS)
    PART_RECORDS  = fast_acc + slow_acc
    PART_CLASS    = ["fast"] * len(fast_acc) + ["slow"] * len(slow_acc)
    PART_PROFILES = {**_fp, **_sp}
    print(f"required stages = {REQUIRED_STAGES}")
    print(f"accepted: fast {len(fast_acc)}/{N_PER_CLASS} (tried {_ft}), "
          f"slow {len(slow_acc)}/{N_PER_CLASS} (tried {_st})")
    print("  gids:", [r["gid"] for r in PART_RECORDS])
    if CORRUPT_SNAPS:
        print("  corrupt snaps:", sorted(CORRUPT_SNAPS))
    if FAIL:
        print("  drop-out funnel (stage:reason -> count):")
        for k, v in sorted(FAIL.items(), key=lambda kv: -kv[1]):
            print(f"     {k:26s} {v}")
    if len(fast_acc) < N_PER_CLASS or len(slow_acc) < N_PER_CLASS:
        print("  -> still short. The funnel above shows the bottleneck. Options: relax "
              "REQUIRED_STAGES (e.g. ['z0']), set RUN_PARTICLES=True so files get made, "
              "or accept that few galaxies have dust at the required stages.")
else:
    PART_RECORDS, PART_CLASS, PART_PROFILES = [], [], {}
    print("RUN_PARTICLES=False and no cached files -> nothing to profile.")

In [ ]:
# 5b. peek at the PartType0 fields actually present (confirm the dust/HI/H2 mapping)
_probe = None
if os.path.isdir(PART_DIR):
    for d in sorted(os.listdir(PART_DIR)):
        dd = os.path.join(PART_DIR, d)
        fs = [f for f in os.listdir(dd) if f.endswith(".h5")] if os.path.isdir(dd) else []
        if fs:
            _probe = os.path.join(dd, fs[0]); break
if _probe:
    print("probe:", _probe)
    with h5py.File(_probe, "r") as f:
        print("groups:", list(f.keys()))
        if "PartType0" in f:
            for k in f["PartType0"].keys():
                print("   PartType0/", k, f["PartType0"][k].shape)
else:
    print("no extracted files to probe yet.")

In [ ]:
# 5e. build profiles for the subset across stages (reads cached extracted files)
PART_PROFILES = {}   # (gid, stage) -> profile dict
if RUN_PARTICLES or os.path.isdir(PART_DIR):
    for r, klass in zip(PART_RECORDS, PART_CLASS):
        for st in PART_STAGES:
            row = r[f"row_{st}"]
            if row < 0:
                continue
            snap = int(snaps_arr[row])
            try:
                gidx = int(_PROG_INDEX[row, r["col"]])
            except NameError:
                # _PROG_INDEX only exists when RUN_PARTICLES rebuilt it; otherwise infer from filename scan
                gidx = None
            path = None
            if gidx is not None:
                path = find_extracted_file(klass, snap, gidx)
            if path is None:
                # fall back: any file in this snap dir matching the class
                d = os.path.join(PART_DIR, f"snap_{snap:03d}")
                if os.path.isdir(d):
                    cands = [os.path.join(d, x) for x in os.listdir(d) if x.endswith(".h5") and klass in x]
                    path = cands[0] if cands else None
            if path and os.path.exists(path):
                try:
                    PART_PROFILES[(r["gid"], st)] = radial_profiles(path)
                except Exception as e:
                    print(f"  profile failed gid={r['gid']} {st}: {e}")
    print(f"built {len(PART_PROFILES)} profiles")
else:
    print("no particle files available; run with RUN_PARTICLES=True on the cluster.")

In [ ]:
# 5f. radial Sigma profiles of dust / HI / H2 across stages, one example per class
def plot_example_profiles():
    fast_recs = [r for r, c in zip(PART_RECORDS, PART_CLASS) if c == "fast"]
    slow_recs = [r for r, c in zip(PART_RECORDS, PART_CLASS) if c == "slow"]
    for klass, recs in [("fast", fast_recs), ("slow", slow_recs)]:
        ex = next((r for r in recs if any((r["gid"], st) in PART_PROFILES for st in PART_STAGES)), None)
        if ex is None:
            continue
        fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)
        for st in PART_STAGES:
            p = PART_PROFILES.get((ex["gid"], st))
            if p is None:
                continue
            axes[0].plot(p["rmid"], p["sigma_dust"], label=st)
            axes[1].plot(p["rmid"], p["sigma_HI"], label=st)
            axes[2].plot(p["rmid"], p["sigma_H2"], label=st)
        for ax, t in zip(axes, [r"$\Sigma_{\rm dust}$", r"$\Sigma_{\rm HI}$", r"$\Sigma_{\rm H_2}$"]):
            ax.set_yscale("log"); ax.set_xlabel("R [kpc]"); ax.set_title(f"{klass}  gid {ex['gid']} — {t}")
            ax.legend(fontsize=8)
        axes[0].set_ylabel(r"$\Sigma$ [$M_\odot$ kpc$^{-2}$]")
        plt.tight_layout(); plt.savefig(os.path.join(PLOTDIR, f"profiles_{klass}_gid{ex['gid']}.png"), dpi=130); plt.show()

if PART_PROFILES:
    plot_example_profiles()

In [ ]:
# 5g. THE key Part-2 figure: dust extension vs evolutionary stage, fast vs slow
def plot_extension_evolution():
    xs = np.arange(len(PART_STAGES))
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    for r, klass in zip(PART_RECORDS, PART_CLASS):
        c = "C3" if klass == "fast" else "C0"
        r50 = [PART_PROFILES.get((r["gid"], st), {}).get("R50_dust", np.nan) for st in PART_STAGES]
        ratio = []
        for st in PART_STAGES:
            p = PART_PROFILES.get((r["gid"], st), {})
            rd, rs = p.get("R50_dust", np.nan), p.get("R50_star", np.nan)
            ratio.append(rd / rs if (rs and np.isfinite(rs) and rs > 0) else np.nan)
        ax[0].plot(xs, r50, "-o", color=c, alpha=0.7)
        ax[1].plot(xs, ratio, "-o", color=c, alpha=0.7)
    for a in ax:
        a.set_xticks(xs); a.set_xticklabels(PART_STAGES, rotation=20)
    ax[0].set_ylabel(r"$R_{50}^{\rm dust}$ [kpc]"); ax[0].set_title("dust extension vs stage")
    ax[1].set_ylabel(r"$R_{50}^{\rm dust}/R_{50}^{\star}$"); ax[1].axhline(1, ls=":", c="k")
    ax[1].set_title("dust vs stellar extension  (>1 = dust more extended)")
    ax[0].plot([], [], "C3-o", label="fast"); ax[0].plot([], [], "C0-o", label="slow"); ax[0].legend()
    plt.tight_layout(); plt.savefig(os.path.join(PLOTDIR, "dust_extension_evolution.png"), dpi=130); plt.show()

if PART_PROFILES:
    plot_extension_evolution()

## 6 · Interpretation — reading the result against the hypothesis

The "ISM equilibrium at large radii" picture makes concrete, falsifiable predictions
that the figures above test directly:

| Diagnostic | Where | Hypothesis prediction (dust survives) |
|---|---|---|
| $M_{\rm dust}/M_\star$, $M_{H_2}/M_\star$ at `post-quench` & `z=0` | §4a–b | **higher / gently declining** for slow quenchers |
| $R_{\rm gas,1/2}/R_{\star,1/2}$ through `QT` | §4c | **stays $\gtrsim 1$** (extended) for slow; **collapses** for fast |
| dust/gas, HI/H₂ | §4d | surviving ISM stays cold/atomic-dominated, not destroyed |
| $R_{50}^{\rm dust}$ vs stage | §5g | **roughly constant / large** for slow; **shrinks** for fast |
| $R_{50}^{\rm dust}/R_{50}^{\star}$ | §5g | **$>1$** for slow (dust at large radii, away from the AGN) |
| $E_{\rm AGN}$, $\Delta M_{\rm BH}$, jet-mode fraction over SFT$\to$QT | §4b | **low** for slow / dust-rich (gas never fed the AGN) |
| residual dust vs $E_{\rm AGN}$ / jet-mode time | §4b | **anti-correlated** (more feeding → more destruction) |

**Logic of the test.** If slow quenchers keep their dust *extended* and *not* funnelled
to the centre, the gas never reaches the densities needed to feed the AGN strongly —
so there is no central heating event to sputter/destroy the grains, and the residual
dust persists to z=0. Fast quenchers, by contrast, should show centrally concentrated
gas at `SFT`→`QT` (small $R_{50}^{\rm dust}$, ratio $<1$), consistent with an inflow
that both quenches *and* destroys the dust.

**Mass dependence.** Every comparison above is split into three z=0 stellar-mass bins
($\log_{10}M_\star$ in 9.5–10.5, 10.5–11, $\geq$11; binned columns/rows in each figure).
This matters because AGN jet-mode feedback in SIMBA switches on around
$M_{\rm BH}\!\sim\!10^{7.5}M_\odot$ (massive galaxies), so the dust-destruction channel
should be strongest in the top bin and the "equilibrium-survival" route most visible at
lower mass. Watch whether the fast/slow degeneracy in the BH histories (§4b-iii/iv)
breaks in a *particular* mass bin even when it holds for the pooled sample.

**Caveats / things to tighten on the cluster:**
- Confirm the CAESAR keys (`masses.HI`, `masses.H2`) and the per-particle molecular
  split field (§0.1, §5b). If no per-particle `fH2` exists, the particle H₂ profile is
  unavailable and the catalogue H₂ (Part 1) is the reference; HI uses the neutral-H proxy.
- The fast/slow boundary (2×10⁸ yr) and the gas floor (`NGAS_MIN`) are config knobs at
  the top — re-run §3–§5 after changing them.
- Increase `N_PER_CLASS` for the particle work once the pipeline is validated on a few.
- For the AGN cross-check (§4b), confirm `BH_CANDIDATES` resolved to real datasets
  (the build cell prints them) and that `bhmdot`/`masses.bh` units match the assumed
  $M_\odot\,{\rm yr}^{-1}$ / $M_\odot$. For a fully independent check, the particle-level
  BH accretion (`PartType5` `BH_Mass`/`BH_Mdot`) and wind/jet tagging used in
  `agn_feedback_view.ipynb` can be folded into §5's extraction.